# A small CNN, no autograd -- digit classifier

Builds on [part-2-build.html](../part-2-build.html) §6 (the delta-blame backprop
formulas) and answers §8, "The dense layer never sees a picture": a conv layer
keeps the 2D grid, looks at small local patches, and reuses the same weights
everywhere. This notebook is that idea, trained end to end:

```
conv(4 filters, 3x3, valid) -> ReLU -> 2x2 maxpool -> flatten(36) -> dense(36,10) -> softmax
```

Data is `sklearn.datasets.load_digits()` -- 8x8 grayscale digits, bundled with
scikit-learn (no download). sklearn is used **only** to fetch that array; every
layer, gradient, and update below is plain numpy -- no autograd, no ML framework.

This is the notebook companion to
[`../examples/02_cnn_digit_classifier.py`](../examples/02_cnn_digit_classifier.py):
same functions, same math, same hyperparameters, split into cells so you can
run and inspect each stage on its own. If you want the whole thing as one
importable file, that script is it.

In [1]:
import time
from pathlib import Path

import numpy as np
from sklearn.datasets import load_digits  # dataset only -- no sklearn model/training code

np.set_printoptions(precision=4, suppress=True)

## Data

`load_digits()` returns 1797 images, each 8x8, pixel values 0-16 (not 0-255 --
this is a small bundled dataset, not real MNIST). 10 classes, roughly balanced.
Divide by 16 to get inputs in [0, 1], same scale discipline as the other
from-scratch examples in this course.

Shuffle once with a seeded RNG, then split off `test_frac` of the data as a
held-out test set. The seed makes the split -- and therefore every number this
notebook prints -- reproducible.

In [2]:
def load_data(test_frac=0.15, seed=0):
    digits = load_digits()
    X = digits.images.astype(np.float64) / 16.0   # pixel range here is 0-16, not 0-255
    y = digits.target.astype(np.int64)

    idx = np.random.default_rng(seed).permutation(X.shape[0])
    X, y = X[idx], y[idx]

    n_test = int(round(X.shape[0] * test_frac))
    X_test, y_test = X[:n_test], y[:n_test]
    X_train, y_train = X[n_test:], y[n_test:]
    return X_train, y_train, X_test, y_test


X_train, y_train, X_test, y_test = load_data()
print(f"train: {X_train.shape[0]} images   test: {X_test.shape[0]} images   image shape: {X_train.shape[1:]}")

train: 1527 images   test: 270 images   image shape: (8, 8)


## Parameters

He init: std = sqrt(2/fan_in), which keeps activation variance roughly
constant across a ReLU network. Each conv output pixel looks at `kh*kw = 9`
inputs (there's 1 input channel here, grayscale). Each dense output looks at
the 36 numbers coming out of the flattened, pooled feature maps
(4 filters x 3x3 pooled grid = 36).

Biases start at zero -- that's fine for biases (unlike weights, see
part-2-build.html §9 for why weights can't all start at zero).

In [3]:
def init_params(seed=0):
    r = np.random.default_rng(seed)
    F, kh, kw = 4, 3, 3

    filters = r.normal(0, np.sqrt(2.0 / (kh * kw)), (F, kh, kw))
    bias = np.zeros(F)

    flat_dim = F * 3 * 3   # 4 filters x 3x3 pooled grid = 36
    W = r.normal(0, np.sqrt(2.0 / flat_dim), (flat_dim, 10))
    b = np.zeros(10)

    return dict(filters=filters, bias=bias, W=W, b=b)


p = init_params()
{k: v.shape for k, v in p.items()}

{'filters': (4, 3, 3), 'bias': (4,), 'W': (36, 10), 'b': (10,)}

## The conv layer, via im2col

A naive convolution is four nested loops: over the batch, over output rows,
over output columns, and over the kernel -- one Python-level scalar multiply
per output pixel per kernel position. That's slow in numpy because every one
of those loops is Python, not vectorized C.

**im2col** sidesteps this by unrolling every `kh x kw` patch the kernel will
ever look at into one row of a matrix. Once every patch is a row, "slide the
kernel and dot it with each patch" is just one matrix multiply: `cols @ W_col.T`.
The only loop left is over the `kh*kw = 9` offsets inside a single kernel --
constant work, independent of image size or batch size. That's what makes it
vectorized: the loop that used to scale with output size now scales with
kernel size only.

`im2col` builds those patch-rows with pure slicing (no copying loop over
batch or output position): for each of the 9 `(di, dj)` offsets inside the
kernel, grab the whole strided slice of the image at once and drop it into
the patches array.

`conv_forward` then reshapes the filters to `(F, kh*kw)`, does the single
matmul, and reshapes the result back to `(B, F, out_h, out_w)`. The cache
holds everything `conv_backward` will need.

In [4]:
def im2col(X, kh, kw, stride=1):
    # X: (B, H, W) -> cols: (B, out_h*out_w, kh*kw)
    B, H, W = X.shape
    out_h = (H - kh) // stride + 1
    out_w = (W - kw) // stride + 1

    patches = np.empty((B, out_h, out_w, kh, kw), dtype=X.dtype)
    for di in range(kh):
        for dj in range(kw):
            patches[:, :, :, di, dj] = X[:, di:di + out_h * stride:stride,
                                             dj:dj + out_w * stride:stride]
    cols = patches.reshape(B, out_h * out_w, kh * kw)
    return cols, out_h, out_w


def conv_forward(X, filters, bias, stride=1):
    # X: (B, H, W), filters: (F, kh, kw), bias: (F,) -> out: (B, F, out_h, out_w)
    B = X.shape[0]
    F, kh, kw = filters.shape
    cols, out_h, out_w = im2col(X, kh, kw, stride)         # (B, N, K), N=out_h*out_w, K=kh*kw
    W_col = filters.reshape(F, kh * kw)                    # (F, K)

    out = cols @ W_col.T + bias                            # (B,N,K)(K,F) + (F,) -> (B,N,F)
    out = out.transpose(0, 2, 1).reshape(B, F, out_h, out_w)

    cache = (X, cols, filters, out_h, out_w, kh, kw, stride)
    return out, cache


# sanity check: conv output shape on one batch of training images
_out, _ = conv_forward(X_train[:5], p["filters"], p["bias"])
print("conv output shape:", _out.shape)   # (5, 4, 6, 6): 8x8 valid-conv'd by 3x3 -> 6x6, 4 filters

conv output shape: (5, 4, 6, 6)


## conv backward: col2im in reverse

Forward turned "slide a kernel over an image" into "multiply cols by W_col".
Backward is the same trick run the other way: the algebra for
`out = cols @ W_col.T + bias` is identical in shape to a dense layer's
`Z = A W^T + b`, just applied per-patch instead of per-sample. So the three
gradients look exactly like the dense-layer backprop formulas from
part-2-build.html §6:

- `dfilters` = delta . activity, summed over batch and patches (`einsum('bnf,bnk->fk', ...)`)
- `dbias` = delta, summed over batch and patches
- `dcols` = delta @ W_col -- the "W^T . delta" term, but landing back in
  patch-space rather than image-space

That last one is the catch: `dcols` has one gradient per patch, but patches
*overlap* (stride 1, kernel 3x3), so several patches touch the same input
pixel. **col2im** is `im2col`'s exact inverse operation -- same 9-offset loop,
same slicing -- except where `im2col` *read* each offset's slice, `col2im`
must *accumulate into* it with `+=`, because a pixel touched by multiple
patches needs all of their contributions added, not the last one overwriting
the rest.

In [5]:
def col2im(dcols, x_shape, kh, kw, out_h, out_w, stride=1):
    # Inverse of im2col. Patches overlap (stride < kernel size here), so a
    # pixel touched by several patches must accumulate (+=), not overwrite.
    B, H, W = x_shape
    dX = np.zeros((B, H, W), dtype=dcols.dtype)
    dpatches = dcols.reshape(B, out_h, out_w, kh, kw)
    for di in range(kh):
        for dj in range(kw):
            dX[:, di:di + out_h * stride:stride, dj:dj + out_w * stride:stride] += dpatches[:, :, :, di, dj]
    return dX


def conv_backward(dOut, cache):
    # dOut: (B, F, out_h, out_w), gradient of the loss w.r.t. the conv output.
    X, cols, filters, out_h, out_w, kh, kw, stride = cache
    B, F, OH, OW = dOut.shape
    W_col = filters.reshape(F, kh * kw)

    dOut_col = dOut.reshape(B, F, OH * OW).transpose(0, 2, 1)   # (B, N, F)

    # out[b,n,f] = sum_k cols[b,n,k] * W_col[f,k] + bias[f]  -- same shape
    # of algebra as the dense layer's Z = A W^T + b, just per-patch.
    dfilters = np.einsum('bnf,bnk->fk', dOut_col, cols).reshape(F, kh, kw)   # dL/dW = delta . activity, summed over batch+patches
    dbias = dOut_col.sum(axis=(0, 1))                                       # dL/db = delta, summed over batch+patches
    dcols = dOut_col @ W_col                                                # (B,N,F)(F,K) -> (B,N,K), the patch-side "W^T delta"

    dX = col2im(dcols, X.shape, kh, kw, out_h, out_w, stride)
    return dfilters, dbias, dX


# sanity check: gradient shapes match parameter shapes
_dOut = np.random.default_rng(0).normal(size=_out.shape)
_dfilters, _dbias, _dX = conv_backward(_dOut, _)
print("dfilters:", _dfilters.shape, " dbias:", _dbias.shape, " dX:", _dX.shape)

dfilters: (4, 3, 3)  dbias: (4,)  dX: (5, 8, 8)


## 2x2 max-pool, stride 2

Pooling has no weights -- it's a fixed downsampling step that keeps the
strongest activation in each 2x2 window and throws the rest away, which is
exactly what makes small shifts in the input roughly invariant to the output.

Forward stacks the four candidate values per window (`pool*pool = 4`) along a
new axis, then `argmax` picks which of the four wins. Crucially, that argmax
*index* is saved in the cache -- it's exactly what backward needs: the
gradient only flows to the single position that was the max, everywhere else
in the window gets zero (nothing else influenced the output, so nothing else
gets blamed).

Backward rebuilds that routing with a boolean mask per offset
(`argmax_idx == k`) and scatters `dOut` through it -- the mask decides,
position by position, whether this offset was the winner in its window.

In [6]:
def maxpool_forward(X, pool=2, stride=2):
    # X: (B, F, H, W) -> out: (B, F, H//stride, W//stride)
    B, F, H, W = X.shape
    out_h, out_w = H // stride, W // stride

    # Stack the pool*pool candidate values per window, then argmax picks
    # which one wins -- that index is exactly what backward needs to route
    # the gradient to the single position that was the max.
    windows = np.stack([
        X[:, :, i:i + out_h * stride:stride, j:j + out_w * stride:stride]
        for i in range(pool) for j in range(pool)
    ], axis=0)                                          # (pool*pool, B, F, out_h, out_w)

    argmax_idx = np.argmax(windows, axis=0)              # (B, F, out_h, out_w)
    out = np.take_along_axis(windows, argmax_idx[None, ...], axis=0)[0]

    cache = (X.shape, argmax_idx, pool, stride)
    return out, cache


def maxpool_backward(dOut, cache):
    x_shape, argmax_idx, pool, stride = cache
    B, F, H, W = x_shape
    out_h, out_w = H // stride, W // stride
    dX = np.zeros(x_shape, dtype=dOut.dtype)

    for k in range(pool * pool):
        i, j = divmod(k, pool)
        mask = (argmax_idx == k)                          # 1 where this offset was the argmax, else 0
        dX[:, :, i:i + out_h * stride:stride, j:j + out_w * stride:stride] += dOut * mask
    return dX


# sanity check: 6x6 conv output pools down to 3x3
_pool_out, _pool_cache = maxpool_forward(np.maximum(0, _out))
print("pool output shape:", _pool_out.shape)   # (5, 4, 3, 3) -> flattens to 36

pool output shape: (5, 4, 3, 3)


## Activations and loss

Same conventions as the dense-only example (`01_mlp_spiral_classifier.py`):
ReLU zeroes out negatives, softmax turns logits into a probability
distribution (subtracting the row max first is the standard log-sum-exp trick
-- it changes none of the math, only stops `exp` from overflowing), and
cross-entropy is the negative log-probability of the correct class, averaged
over the batch, with a tiny epsilon so `log(0)` never happens.

In [7]:
def relu(z):
    return np.maximum(0, z)


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)   # log-sum-exp trick: stops overflow, changes nothing mathematically
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def cross_entropy_loss(probs, y):
    B = y.shape[0]
    return -np.mean(np.log(probs[np.arange(B), y] + 1e-12))   # +1e-12: never log(0)

## Forward: conv -> relu -> pool -> flatten -> dense -> softmax

Wires the pieces above into the full pipeline. Every intermediate that
`backward` will need -- the conv cache, the ReLU mask, the pool cache, the
flattened activations, the final probabilities -- gets stashed in `cache`.
Nothing here is recomputed on the backward pass; it's all just replayed in
reverse.

In [8]:
def forward(X, params):
    # X: (B, 8, 8). Save every intermediate -- backward needs all of it.
    conv_out, conv_cache = conv_forward(X, params["filters"], params["bias"])   # (B,4,6,6)
    relu_mask = conv_out > 0
    relu_out = relu(conv_out)
    pool_out, pool_cache = maxpool_forward(relu_out)                            # (B,4,3,3)

    B = X.shape[0]
    flat = pool_out.reshape(B, -1)                                              # (B,36)
    logits = flat @ params["W"] + params["b"]                                   # (B,36)(36,10) + (10,)
    probs = softmax(logits)

    cache = dict(conv_cache=conv_cache, relu_mask=relu_mask, pool_cache=pool_cache,
                 flat=flat, pool_shape=pool_out.shape, probs=probs)
    return probs, cache


# sanity check: probabilities sum to 1 per row
_probs, _ = forward(X_train[:5], p)
print("probs shape:", _probs.shape, " row sums:", _probs.sum(axis=1))

probs shape: (5, 10)  row sums: [1. 1. 1. 1. 1.]


## Backward: the same pipeline, run in reverse

`softmax` + `cross_entropy` collapse to exactly `(predicted - actual)` -- the
same identity used in part-2-build.html §6/§7, so `dlogits` needs no chain
rule of its own. From there it's mechanical: dense-layer backprop to get
`dW`, `db`, and `dflat`; reshape `dflat` back into the pooled feature-map
shape; `maxpool_backward` routes gradient to the winning pixels; the ReLU
mask zeroes out anywhere the pre-activation was `<= 0` (ReLU blocks blame,
same rule everywhere it appears); and `conv_backward` finishes the job.

In [9]:
def backward(params, cache, y):
    probs = cache["probs"]
    B = y.shape[0]
    Y = np.zeros_like(probs)
    Y[np.arange(B), y] = 1.0

    # softmax + cross-entropy collapse to exactly (predicted - actual), same identity as SS6/SS7
    dlogits = (probs - Y) / B

    dW = cache["flat"].T @ dlogits            # dL/dW = delta . activity, summed over the batch
    db = dlogits.sum(axis=0)
    dflat = dlogits @ params["W"].T           # W^T delta, wired backwards through the dense layer
    dpool_out = dflat.reshape(cache["pool_shape"])

    drelu_out = maxpool_backward(dpool_out, cache["pool_cache"])
    dconv_out = drelu_out * cache["relu_mask"]   # ReLU blocks blame wherever the pre-activation was <= 0
    dfilters, dbias, _dX = conv_backward(dconv_out, cache["conv_cache"])

    return dict(filters=dfilters, bias=dbias, W=dW, b=db)


# sanity check: gradient dict has one entry per parameter, matching shapes
_grads = backward(p, _, y_train[:5])
{k: v.shape for k, v in _grads.items()}

{'filters': (4, 3, 3), 'bias': (4,), 'W': (36, 10), 'b': (10,)}

## Training loop

Plain mini-batch SGD: reshuffle each epoch, slice out batches, forward,
backward, subtract `lr * grad` from every parameter. No momentum, no Adam,
no learning-rate schedule -- deliberately, so there's nothing here except the
gradients derived above actually being used.

In [10]:
def train(X_train, y_train, epochs=15, lr=0.05, batch_size=32, seed=0, verbose=True):
    params = init_params(seed=seed)
    r = np.random.default_rng(seed)
    n = X_train.shape[0]

    t0 = time.time()
    for ep in range(epochs):
        order = r.permutation(n)     # reshuffle each epoch
        for s in range(0, n, batch_size):
            idx = order[s:s + batch_size]
            Xb, yb = X_train[idx], y_train[idx]

            _, cache = forward(Xb, params)
            grads = backward(params, cache, yb)
            for key in params:
                params[key] -= lr * grads[key]

        if verbose:
            train_probs, _ = forward(X_train, params)
            loss = cross_entropy_loss(train_probs, y_train)
            acc = (train_probs.argmax(axis=1) == y_train).mean()
            print(f"epoch {ep + 1:2d}/{epochs}   loss {loss:.4f}   train acc {acc:.4f}")

    if verbose:
        print(f"training took {time.time() - t0:.1f}s")

    return params


def predict(X, params):
    probs, _ = forward(X, params)
    return probs.argmax(axis=1), probs

## Run it

Same hyperparameters as the script's `__main__` block: 15 epochs, lr 0.05,
batch size 32. On 1527 training images of 8x8 this is a few seconds of plain
numpy -- no GPU, no batching tricks beyond what's already in `im2col`.

In [11]:
params = train(X_train, y_train, epochs=15, lr=0.05, batch_size=32)

preds, _ = predict(X_test, params)
test_acc = (preds == y_test).mean()
print(f"\ntest accuracy: {test_acc:.4f}")

epoch  1/15   loss 2.2700   train acc 0.2141
epoch  2/15   loss 2.1680   train acc 0.3052
epoch  3/15   loss 1.9540   train acc 0.4139
epoch  4/15   loss 1.6121   train acc 0.6038


epoch  5/15   loss 1.1820   train acc 0.7485


epoch  6/15   loss 0.8372   train acc 0.8284


epoch  7/15   loss 0.6204   train acc 0.8664
epoch  8/15   loss 0.4972   train acc 0.8841


epoch  9/15   loss 0.4250   train acc 0.8978
epoch 10/15   loss 0.3634   train acc 0.9181
epoch 11/15   loss 0.3227   train acc 0.9306
epoch 12/15   loss 0.2905   train acc 0.9358


epoch 13/15   loss 0.2742   train acc 0.9332


epoch 14/15   loss 0.2473   train acc 0.9443


epoch 15/15   loss 0.2500   train acc 0.9319
training took 0.4s

test accuracy: 0.9000


## Gradient check

The correctness proof, not part of normal training: compare the analytic
gradients from `backward` against central-difference numerical gradients
(`(loss(w+eps) - loss(w-eps)) / (2*eps)`) on a handful of entries from each
parameter array. If `backward` has a sign error, a wrong axis in a reshape,
or a transpose swapped anywhere in the conv/pool machinery above, this is
what catches it -- unlike a training curve going down, which can look
plausible even with a subtly wrong gradient.

In the standalone script this runs behind a `--gradcheck` flag; here it's
just a normal cell.

In [12]:
def gradient_check(eps=1e-5, n_samples=3, seed=0):
    X_train, y_train, _, _ = load_data(seed=seed)
    X, y = X_train[:n_samples], y_train[:n_samples]
    gc_params = init_params(seed=seed)

    probs, cache = forward(X, gc_params)
    grads = backward(gc_params, cache, y)

    def loss_of(prm):
        pr, _ = forward(X, prm)
        return cross_entropy_loss(pr, y)

    def check(name, coords):
        arr, g = gc_params[name], grads[name]
        worst = 0.0
        for idx in coords:
            orig = arr[idx]
            arr[idx] = orig + eps
            lp = loss_of(gc_params)
            arr[idx] = orig - eps
            lm = loss_of(gc_params)
            arr[idx] = orig

            numerical = (lp - lm) / (2 * eps)
            analytic = g[idx]
            rel_err = abs(numerical - analytic) / max(abs(numerical), abs(analytic), 1e-8)
            worst = max(worst, rel_err)
            status = "OK" if rel_err < 1e-4 else "MISMATCH"
            print(f"  {name}{idx}: analytic={analytic: .6e}  numeric={numerical: .6e}"
                  f"  rel_err={rel_err:.2e}  [{status}]")
        return worst

    print("gradient check (eps=1e-5, central difference, n_samples=%d)" % n_samples)
    w1 = check("filters", [(0, 0, 0), (1, 1, 1), (2, 0, 2), (3, 2, 2)])
    w2 = check("bias", [(0,), (2,), (3,)])
    w3 = check("W", [(0, 0), (17, 5), (35, 9), (9, 3)])
    w4 = check("b", [(0,), (5,), (9,)])
    worst = max(w1, w2, w3, w4)
    print(f"\nworst relative error across all checked entries: {worst:.2e}")
    print("PASS" if worst < 1e-4 else "FAIL -- backprop has a bug")
    return worst


_ = gradient_check()

gradient check (eps=1e-5, central difference, n_samples=3)
  filters(0, 0, 0): analytic= 1.685295e-02  numeric= 1.685295e-02  rel_err=9.03e-10  [OK]
  filters(1, 1, 1): analytic= 0.000000e+00  numeric= 0.000000e+00  rel_err=0.00e+00  [OK]
  filters(2, 0, 2): analytic= 1.957751e-01  numeric= 1.957751e-01  rel_err=5.56e-11  [OK]
  filters(3, 2, 2): analytic= 3.252274e-01  numeric= 3.252274e-01  rel_err=2.14e-11  [OK]
  bias(0,): analytic= 5.386819e-03  numeric= 5.386819e-03  rel_err=2.69e-10  [OK]
  bias(2,): analytic= 2.470956e-01  numeric= 2.470956e-01  rel_err=2.42e-11  [OK]
  bias(3,): analytic= 5.588757e-01  numeric= 5.588757e-01  rel_err=3.49e-12  [OK]
  W(0, 0): analytic= 3.147896e-02  numeric= 3.147896e-02  rel_err=5.83e-10  [OK]
  W(17, 5): analytic= 0.000000e+00  numeric= 0.000000e+00  rel_err=0.00e+00  [OK]
  W(35, 9): analytic= 0.000000e+00  numeric= 0.000000e+00  rel_err=0.00e+00  [OK]
  W(9, 3): analytic= 0.000000e+00  numeric= 0.000000e+00  rel_err=0.00e+00  [OK]
  b(0,): 

## Inference, with a save/load round-trip

Same demo as the script: save the trained parameters to disk with
`np.savez`, load them back into a fresh dict, and run prediction on the
first 8 test images -- proving the weights actually round-trip through disk,
not just staying valid because they never left memory.

In [13]:
import tempfile

weights_path = Path(tempfile.gettempdir()) / "02_cnn_weights_notebook.npz"
np.savez(weights_path, **params)

loaded = np.load(weights_path)
loaded_params = {k: loaded[k] for k in loaded.files}

sample_preds, _ = predict(X_test[:8], loaded_params)
print(f"weights saved to and reloaded from {weights_path}\n")
print("sample predictions (loaded back from disk, to prove the save/load round-trip):")
for i in range(8):
    mark = "" if sample_preds[i] == y_test[i] else "  <-- wrong"
    print(f"  predicted {sample_preds[i]}   actual {y_test[i]}{mark}")

weights saved to and reloaded from /var/folders/jg/kvcwx7yx15d2y07qr4gf1kj00000gn/T/02_cnn_weights_notebook.npz

sample predictions (loaded back from disk, to prove the save/load round-trip):
  predicted 6   actual 6
  predicted 6   actual 6
  predicted 6   actual 6
  predicted 2   actual 2
  predicted 5   actual 5
  predicted 6   actual 6
  predicted 6   actual 6
  predicted 2   actual 2
